# Qwen weight transfer, MOHAWK alignment, and recovery

Package implementation: `src/elastic_embed/mohawk.py`. See [the bundled guide](../README.md).


Use a BF16-capable GPU with room for the **8B** teacher (A100 is the intended first debugging target). The runner does not silently quantize or replace the teacher. A T4 is not the default BF16 path. CPU offload is configurable but needs host RAM and will be slow.

Every command runs in its own Python process so the teacher is released before student training. Keep the same config and Drive run directory across notebooks. If pip upgrades packages already imported by Colab, restart the runtime and rerun setup.


In [1]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
# Upload the elastic_embed_colab folder to My Drive. Change this if nested elsewhere.
BUNDLE = Path('/content/drive/MyDrive/elastic_embed_colab')
assert (BUNDLE / 'pyproject.toml').is_file(), f'Upload the complete bundle to {BUNDLE}'
os.chdir(BUNDLE)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Keep model downloads on local Colab disk; caches/checkpoints go to Drive.
os.environ.setdefault('HF_HOME', '/content/hf-cache')
subprocess.run(['bash', str(BUNDLE / 'scripts/bootstrap_colab.sh')],
               env={**os.environ, 'COLAB_PYTHON': sys.executable}, check=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


CompletedProcess(args=['bash', '/content/drive/MyDrive/elastic_embed_colab/scripts/bootstrap_colab.sh'], returncode=0)

In [2]:
import yaml
CONFIG = BUNDLE / 'runs/smoke-mohawk-config.yaml'
CONFIG.parent.mkdir(parents=True, exist_ok=True)
if not CONFIG.exists():
    cfg = yaml.safe_load((BUNDLE / 'configs/smoke.yaml').read_text())
    # Set these BEFORE prepare/cache/train; resume checks enforce matching identities.
    cfg['run_dir'] = str(BUNDLE / 'runs/smoke-mohawk')
    cfg['data']['sources'] = str(BUNDLE / 'configs/sources.yaml')
    # Default smoke budget is 20k. For a smaller wiring check, explicitly lower both:
    # cfg['data']['max_examples'] = 1800
    # cfg['data']['min_per_language'] = 20
    # cfg['training'].update(full_depth_steps=5, prefix_steps=5, matryoshka_steps=5, effective_batch=8)
    # cfg['mohawk'].update(examples=32, validation_examples=18, matrix_steps=3, block_steps=5, effective_batch=8, block_freeze_ffn_steps=2)
    CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(CONFIG.read_text())

def run(command, *args):
    import sys, time
    log_dir = CONFIG.parent / 'logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f'{command}-{time.time_ns()}.log'
    argv = [sys.executable, '-u', '-m', 'elastic_embed.cli', command, '--config', str(CONFIG), *args]
    print(f'Python: {sys.executable}\nLog: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f'{command} failed (exit {returncode}). See output above or {log_path}')


scale: smoke
seed: 42
run_dir: /content/drive/MyDrive/elastic_embed_colab/runs/smoke-mohawk
teacher:
  name: Qwen/Qwen3-Embedding-8B
  revision: 1d8ad4ca9b3dd8059ad90a75d4983776a23d44af
  precision: bfloat16
  batch_size: 2
  device_map: auto
  max_memory: null
model:
  backend: fla
  vocab_size: 151665
  input_rank: 256
  width: 384
  layers: 12
  heads: 6
  head_dim: 64
  conv_size: 4
  ffn_width: 1024
  depths:
  - 1
  - 3
  - 6
  - 9
  - 12
  dims:
  - 64
  - 128
  - 192
  - 256
  - 384
data:
  sources: /content/drive/MyDrive/elastic_embed_colab/configs/sources.yaml
  max_examples: 20000
  min_per_language: 100
  english_cap: 0.3
  long_fraction: 0.15
  default_length: 256
  long_length: 512
  validation_fraction: 0.05
  test_fraction: 0.05
  exclude_jsonl: null
cache:
  shard_size: 256
  audit_examples: 128
initialization:
  mode: table-ffn
  svd_rows: 16384
  ffn_score: weight-rms
mohawk:
  enabled: true
  teacher_layers:
  - 3
  - 6
  - 9
  - 12
  - 15
  - 18
  - 21
  - 24
  - 2

### Update the Drive copy from this notebook
Run the next cell to apply the bundled runner fixes and independent sanity-check script directly through this notebook’s Colab kernel. It backs up replaced source files and updates the output/logging helper. Configs, caches, and checkpoints are unchanged. This cell works even if only this notebook was updated locally.


In [ ]:
# Apply the bundled runner patch to the connected Colab Drive.
import base64, json, subprocess, time, zlib
from pathlib import Path

BUNDLE = Path(globals().get('BUNDLE', '/content/drive/MyDrive/elastic_embed_colab'))
assert (BUNDLE / 'pyproject.toml').is_file(), f'Check the mounted bundle path: {BUNDLE}'
_patch_files = json.loads(zlib.decompress(base64.b64decode('eJzdfQt3Gzey5l/BaDaXTYdsiXo4Nj1MoiT2jO8kttd2ZnaOxNOnSTbFjshuuh96xFf727e+KgCNflCSZ2b3nrOciUU2gEKhUKgXCujPe3k234/WYV7E8yDazKLFfh4mcXHrb2/3xmrvnP/3YROu1yosi1WaRQuVR5swoQZqEYcXSYq2+QuVpKqIwvkqytR1FF+sinygijC7iPAlzVR0U0RZEq7VIizCPCp8gX2exJttmhWKqm7DLI/sg3maFNRoHc/so9/yNDlPllm6UduwWFGR0iXv6KetVsSbCkyRhfNoFs4vz5PqWZrNVxoQf/eTxEBalsm8iFNgGubqla7lEzbL+MJUWqfhIpBHAxUvIiJHcWuqYoDK9nQZJfHvUWYK49QUhUW6IapjTANFdJtf5uXGVCOs4yRO6h3mRYmuMJA/qk9llN0OVBatoyuaDiJInocX0UDl4SYaFumWJmgVZguVRBdhEV9RSZlQ7bCIqmc+IP0UESGjjArWt86c0oyFm+06oulLUkI3UelyGc9jIgxNZ14QAFQj1rhV2/Q6AmvMomS+2oTZJQH+8fTDyw9qos7OE0UfrzcPk3m07g1U7y/ptVqk6rWSR2pzq/Jyls+zeAvSf4c6H1MVJQt1m5YZjS1eqA34M8tX8Zb4aRslxIdFQSTKCTUw5TqaFxogDZLA+BpMFiXR9ZcC4kbhWmCsIjULL4neKF3naplF+UrNsihcKPC82qQZZsvv9QdmtFlEnLRA8x+JdK8VLQSMc5Mm0a0CP6olkTEkKGARFRfRhsf9KizXBVG0JLakdZLTRIfreHmrqwtY4itVEFZpFl/EYNVteLsh1qCx0SJd+J2AiDY0QQRhu6YlsRioWVloeDT0LOJppjnGTNpxMyOqdRReRRg3Vh5Rl3DZEC2uosQdMljwOs140K+B70XKYzbPfYWJnzM58viCBi1T/Wse8WhMPUJqnl6B3Os4ucRYM+JXQiOcz6M8xwOeTfqZlknBuP4jLRkyjTW5iPRsZ+kyXhPceF6UNL5wSSKIe+aVJQzyHoDjnLh9S/NOPFyssrS8WKVEHCDFjZI0rY10QUsGCALA30nkRYDAI82I/b5TrwtaIDEJTaK8rJ4sowaarBlmnzHcxHkOXMCMPH8l/yyE8PNL/EhKsCuhK02IPvSLRHC8NgPPdL/qmnrTqEFOl0yuZYkVehvlNI5FeKtnlvDN03JLkiDC7BOli4JIRYwckQRZF+5oZ2FR6MH+QmNcEQuTnCB+EIL+VuYsHohkmeUbmoYFrSul26oF+ChXWbiNF4QOVpp0TWuJRBkNnWYuu+BFtAME1FTOywCcl7Equc2JpYvrKEqkfZTb5ss0LWZQXJuwoCVKwoRoAiYCLtfu+K7jZawHtw63JDrBSZg1kvAJhIFebXECVogKPYtpOSPpuKJuuHSebrYl6EE1iIsvTWuSRDxckhmkD6XudQy5TaxMQKiJxdnCoOnRzQlptCYxcc28cR3NchIWYKsMevnWNt6UeVRuaGZyWqK3kGbzGITdpkw/d8T5Ooq2skwv6RsxDrMa8QMRCAhu4sWC2CFd8q8E+tzXJOK2QJAESQRtQtPJlMnKrcH2taEgDZkmm5v40p80J95L0FAvNu4FWjRzO+NBRVjRpI1kYS+45iwjaUxMv8jdQSVpnEemXQIbZJZmItkwSAxwQ+v5Gm2v42JFSrVcgGrxXMaGbyLjeOHToip4USZkhpC+SnlaiLwkTGYMTRPDb3e6InlJg0+3W0IZ08H1uQPQJy4A6lMZa2Z6jToJS5+U2DWl6dUTcUEqvC5l16Sd52HBejTELN9iPeZYAtAvWqJQDZatbyEdaBEtSSzwss1DwolosnhBfUakGNjQCueQPSpeQsionFhGrTRfvnXkFEFlkTZDb0vMoQyHhEm4hPmwSjcRtzpVULp5uiyuMQHllqwilpLhFUmukFZOTcKUOZMhl+klTLIyYSkNe+U7q4SpII+yq3jOoEjUhbea464hbUXKEnarGJqcjKpFuY78Hc1NF7Q+YTES6SvxGBFdaGlHwj4WZ8zKbF3WcP8tnckUhhviqJQXEoQUTcYyzkg0UgWG+iGKuJBYKrsdktlGEjsiCysV1c0sSVb0Nouu4pRQZSECtZTRMp7LMP4O/U18A6S22zWZX0nBCpEZrqBZuaXJzbFwof1IxIgp2wADaoBUmVoSyhCMIspXpOdvFUT1kkRL3ZxJCisjaelCTYOGYWw1FqqoEIra9kE4UL9khaTXCcAzvybEN3OetrRYsWabZ8yIRFwyj8iuBx+JJtR2TX4fROLlcq4lg2kPXBypSv1lLIwyohbxMDsoRDYta6ph0qij7OJWJhQKCmoEoENIgEiUOS8YQpbWUVJq3N7JWiL0rsgfICUjhUNSbxBiNAWzaB6ShMbUhAJVd1ZpZK6noecGPLO3fFdpvHYowVJXSFCSdY5ymvZwi4Xrau/5Zbk1ZrdYX4RmAcW2YXVepLzyN0a3w7C9zHnV/RXCOlQCg1Db3qIPsa7EqsoV6zeZykXEi4v9gzQjd8SsN8aaHI2IhAB0cgUQdkQLJAthF2TFAHMSNGpOQpCqEavTXFP/H0gSkcpzh72JosJY39dgDFofbHLRc+bYFLI2zUivfudMIPWuxQbXpkWWEpl5gWJRF2xYrNfpNWAYefG2UXMDiyiDjZaIVOE51CpxJshUJvYlmV4R26BGUbAoCKlpQ2mTBkSj5GJNiECWbres2shYyEKyISEvU+1hJWRurMlBFFXGooJNEUdrayqWibF9HWPCbQLxXrD4E7MNcuO+tmzplBm8Ukw/JuoGXhnrUmL8dVSbKXIiL6OCFcCpUWbrNQYTkiaC/JUaMgGyZr+TFXqNPmgmZyVIjBhDFqpwweqKEIdDoacNFj65K8a+fC0Cc1cbXu8VeOBTWSTwOXhmSfitxVITg5ue5UlaNy2J1sTFaPsz6Yac7OxLcXawQDbqMklpaAtZvFgBcKRtV+yKZuRzF3CQaUZIpVCfVH9RZuw3YC7ImpFeuN1Lp42wIEStVWAN74axsP2tY7KsyNJmOc0KfJtGBbu9c/GoHWeP9Ec415KSnHmwNtMrJ7qQpZZB10baTNqSzSJL+D3xBmFXyILPU44nGEctT2kxwzvYQo6y8MVSAyzIrauYMBJLQoD9AlOuC9CWTHMytQgeCMQ81QkAg6aOaC4JUs76cxOToI1EpxJZqXptDabrBdGOeTXjiYQmTkChkET1OlpWcoLsBxEtPyK4o65XEQu0itU047OJIe5iNU8GCAlzONuh0dynqNWC0FFbm6CEL5ir0horIcaGHCieZ2II0lC0NnOzKqcICv2YpXk+hMQpWYzDIpQglSq1u45Ak3pJwijOVyDvImbCqlCsOFOCQBUJef88+fj+9M2Hn08/vn77phYYinjtV+Gh8/LgYLb8EX+WRxuyiMqIGEzKQxijtBg4WDSPpU7ynTNFS15tTizifO8/z8vDg9HzMFbw2bhR9By2Av1Hel0iD5Gvfkw3Oopiva9MfXe+5zr+AF5z/1MSAWhCq/8dcVHxnXqZ8yNa2mTQxfMVvOOL6JKB15YQ85HjXZ/C1yVqhVhQm6hUNOASQw7n4SwkY5FWRpQQfcsNTR4pWBfain1YxzugYT4fneDf4wX+PVzi36NIVQVH8ugZVzrggkP58Y38iOQH1zpwao2e8o8Z/j14xo+OnvOjkfPocO50clhhor8fOJgcz6TF06pD6UPjq6E/c2c65Jk23juVPz0+xr+HjPzT4xD/Hh3xI/n+jfP8OVc6XEiLqurxM37yTKqqqliAH59UzeS7NDg+blWV77rrE7dUVV3UGgi8p06zkfuk5naEzDziOJ2XRweLA/w7e45/nxLJnx8AvaODZ3N+8hT/opujg28i/HvCNTGFy+Vo6UD+fQWQVfzyvDyOviHgJ4dgARoJAT/55pj+/ebpMwJ4cgL8To7nhOvJUbRE/egECByg/uwoQunoG6ejKWLY58kiWpKPdAObz+uPpX/E/PMB6dIFzD6IielAfb6TQjQIFwsPlUwDfMh6xSNWQgjnSWOnAleSp2eoOSXAJPMYUN6v1+NnPtzkRPdUlWcRIZvUQUnpnGOshC6VFF68mFxGZH1xlH4ClD/1B1CDMcLu/GBLDxCg5x8r+kH2p1SN+g2MWL4LPAJCzagyxskhdt3/7DaIF9T/ZzJoz3rxojcdK/rKTfGXqjOKmpJQ0rnEySusjbif4MtO5LmnM0JnetYzD3vTJsoO5gJN8Acerh6w1LPqgyaGpKcHhHz4AdQu9zCoy2lrMEIY/PQqVIhlQVf8BUl7fUNOPXmawxjAoEaIgYNHxaHUjBTYVbj2rkgjpFk+MDqtIkqtqeHMfJ5mPCDd7kw3m6rv7aOq1dT/qKeGd82omWxNkVGSp5nn1PSJ/aIbYiAePsfsLB5mHqjrDB7eRGNxJsBCts08sL5Gpt83+3R6Kkj3EynIbucdIjIxJBILg3xMVuGcTLJwm5NxtkkX2Lspc1l314iNIBRIuEmU2tdkD5NLUMHT9Ph2YrA7Gw/UG7JHpmqoRtHwm76flxtvVJ+uzz2NaW+sXLzFZNAk0WXOHAzq/Ejqsfh+RNWW6zQsPE+Q+tNEjfq+POr75H4nXr+z5UlHy5MHW26yrGo3UvtCCtNqZzOqtUg3gcEYDZuDq1UCcmS2eieDZr1+u+ldxdg8HzSe6wcXg+FoxJwxlQig4zut0s1kaCaMpDA7uGm69oTd4nwZJ3HBnfR9cvxptIgO7KrG8E3FcUvsfu5h27HEbPcS7L6iEdb5ch1e4OlZ7415rHg7e4F9vTExJS2NIr5ADHIWLbHJsSwzNsfNTqvfm2rRqJcmDfOVL0H2+HcegZm4gaoN+yILN9Uir9a2Wc/pcknFqHX2v2W80a0sQVCFgBW322giJSBKXy/EfMsbjBPDQKaDoe2A+eeAGCn/VIbQobx+CDHDWQKIqcOa1M4TE1/Irf6EtfeMqJ7c1qnO7Ywi7L2Jwmz4e5Sl8Ce25D8aLHrV9GuUGeLxvZAQfiAmK0KOCWgiLwzMMQRaHsMRMvLGdkO291KThChrRqq+Vf7zk51d/g0bSKv4YkUuViSBqXhDrlYGNzmcw9nhyH6ckMwkL3IWiZp4QayTI8BBJnycp0WWbm8tJouoIHfNpWxTPzn4GLVNlQV5ZghW1qxne2TmiJab9h+n1qYVcI2JGe5n6H9W/2IJwB/SYII5fUkiKt1qgNWTlQZdPYkG3VqdxRRX3fCmHcARX0YGZO3p6kEgWh1w/W+JRl+Df55W+LmlK116pycB8w+bIdzMFiGMjLHCImBSkoniTommkhGK5qflXmalOkZ9CHp/N1+9pMpVQgzEO28tc5CZOp5Fa+r65OCrio9gOl7Fi5KD8pJgYRmqbVaAXTLLITIYy100ryisbLBmeUOPVsIzvXSlJv9tK6HrAAsz2PBECs+ywPCha/qig0wdolyz0i7dZpqEN80W4Y2AreRBIAKFKsqXJjQSAAEyVrDHobm2josjIQbd1fNiUatNvzvQZqYI5/MyQ8xr3Mkqhl9b1Wpc3DYSqE5QX0sOdP0MwLlifXk58G3FJsGN9QqB8MWWbBMaS8rA+AnBA8A7Wde1Mab3iIbdn11MX4f8BcPYhnHG5kPdC6JuzHfIUGGYiSu9Xdnt4jRteXC7Po8iS89ICsJRSy3HjAPNSxpmwPE53oXw5kvyuJAjNzDujuyjDHjnhUQXvhTzVZDTMjM6X1tGE27oQ1p4RsjC1B9w0hfK3Ww0tysNWzfKSdUuApMUOJHWPjkaXk9S53pw7PvyRFeTR5UyJREuajAJN5GEJa5ihD57DfdfV6z3yflc7hNRCX8gB2R5cWb7nPLjRrQAO5uR+huRNnqZZeSASRiXyVul9dnkP7IekDmlt64kiYzsGT1QR8eAkj53f5XOwxlPQK/CicsJo1qpawnfhxa3QcDwcejQKNjL1GPBTPbh/71FDL4aIiLqSPDkfR9MvcR3DZFlRHqAyRY7lgzU04y3DdkKn3zMyggeRpnIttLkVbjOo4FWT4G4ufkEUdGKXAKQrBj8DeJF3pv6+SrcRmejKRkDhydP76XNB05mNTEm3oZCzhE1k/HRsBDnkhysmDeoKgR7LvP7RerJEur7LOR0IamlOTOkMkyl/QnMoS2kCabBSHtftqGxx0cLoVyEZGRERAnoY5IsR4c9O/oKuA5sebYOYn/8ffS0thTaNPg1wdZRmmH+K4Q0+mLKY8Y+L0j+iTBdyE68ZdRFtC1WufWPJI1pwkkTPnYcAk7DIwbSQHn7Unt2CdIJkzmpZYLn1bwL5NOgJwlJHAycgFyXdDKfeXFjYyNhWRDL54UmZMOX0vTp10k5mTikE8pXqcZ+Uq7X+qfXkOE8LOp93BbtWZSXa9CEqeZ1cO1ZPI6/rgY1dfjpYR1o4GGnIAFrkrbPL+8F2sCdpxXTOFDQFhw9FKR9ZJ7mXpPKDnOcccOpsXm5vY18zLel9TMjHRjqZA1yBJhxjNVehGC6XqWzYOQVmce6h1BL11eRWG203mnBUrFJkuY6XFJANY9dzSKPqAxBgBhWpF7LTZWaF9G22ZSeAGyRFmQf4mder+EWtG2KFCq6aOHDiXYHfYS9doBCcROaTCPre9HbPa1c6ZFV4Y6AGVccDi8umacQukEeETMvMAw9Oc1+SLCgl94vb/9y+ve/Ihsz+j0SAZ8Tr2HpKlje2DCGOl3GJC1m6xQbgQj1gSWo3vrWV6+LvGqjY32cPUUI55x+pjPJpf2QJuci4f02MtEKv8fLVCyPaFOTkpt0FV5fGjmJiKEWRZK11GGwEChPzBl6HMRJOlDVT6wV9/cG7Bok5BiSFvYYpm1lf0kj+9O2acne98TwVNbWziaDUe/da7ONJgxKCBk8MhrOPbD5/8zPCiGtfEX2Xz00ilVEBhOvz7EWiaS7ZInm/daq56xCvarNur8b1ER9vCFhVdmWyB+wsoHPZ4Dc5qyGf5pdlJjBd1xCssfm8E+CYJHOg6DvNvXJFAhC3cbrDYfWDtRJbQu2Eu5vwxTBoknoKdkLX/dwfCLFfszkrFr2EUdKhHOGm5B8kxvnAXOg2JOSas7+MMn+IdOrJ2trqdtQ69s0X12GvfscFqIXUu4nZ61O6oBdcPcPVQuBgQXdg8Lr3d/IyAmXKtxMNvVEHWv3H47Ll8EmXTNko5RMOahaYuwKxPH9rYXzTBeYPV66XJf/oHbuLGH89Cv9hriiG1GTlpEsNBc52ZSYRdapNX0Sjxv3RVjP4y7ku43qcQ7bRGmjgjcXZWnRTz/OA5ul4RnLjbRgr28xNhDI0GCaSiWnSLtbMlHcUzUxlaHIAABZGyk198DM8rTvEErD6+hXl9W2WB+I+SPwbPZmjdThs0hQ27ok4Dij8YG8M96QHNQ3+Fijoke9R6MtvOY66jE6VEfD6Ll40WMXJrZ+rFxF4ee7Nrx1vIkL2/ysd6rPr5Eby+fZbNSOw718gq1+yskGNJxjTb2HjLXeX9wzV/bQjemMT0vp0PMLheAgn9EyISMVrvlwhck05eBmFQF/sPM31SE8ffYOUUjZqNOn8pC+s3gheVacMhQnCeeda3fvcb1clNhJLCJJZ53lrPPleBDIppUX9c3ZWWQLbEuisE4TkzNFSF9CWp5mpUf0+3a5HJrwXXUADUptvi4XsFPkdBtzGNKQrqM1n+KrDr7J3v7DXX2MtfHCdgw7znZLFLbOLyTuYmyJIo6bc3AGUV+mKmaQw0JIx1ImgvSITl9qqw3ms5xQ4nHlnDEb3cQ6FzjODavyPsoQJZwnnxADOaxq/DW9XTPhQ5Ii8ORRJTp0FZYXXIuFDdkkwSLOIGP2kfvGzvSwp74W+56s9SW+eL2v/vHV5qvF8Ku/fPXLVx9IEH6tej7ONdotG4hoEi4mUgWZR/7egXGhEbESy71OEVaZ+2STAAYi7UqWC/3B3qj8OVFmq4x00nJd5ivXiFjKWcUL9nh4rGI/1PQI+XWTneOW1hhSFZnAR+92sulJU5AXbSdKBOZZTVhNzxjglMVoFZHXJyx6NdHmuET9uzpoIdqy95mh3WnnSAxNXo/FC5Vfxjh/0kUW84GrGydlVD0l26QxCrEybVAXJiIh/09GHRsqvd/u6sxQhWwT+X2RYROL+pR9i88NUvDODeqxL9OM41XEcivdQxI9r1LbOnW7wS57fz99/+b1mz+TGpLJ4P0fJAmZ1s7k3DcZjrmehbxjZbZYd/vo3IYWCLa3aGI6ami6ZpCUk2rPH7kXY2oyfXjzfydMOzdnxIiM+d3+Z4J5x7OlO+1uLZOG8mq+4X5hincMwiW5oXStV5oAA/N8T4Ce703v7iO5+2mvhQbCnp4c6qsPgp95o8HTY7Jv6O/RM/5yNBgdPsOXp4PR80MuOjR1DrlSFyN1j248emYGOD6UIY7/dHTHNZ2B1ramaLxj/+Cru3qV2raUqdJ7KOy07NWhWHuIIAAoRHB3fx01T7prtrb6uNrx8lGTBt7H3iWHKw0zyabmY6is1C4uAox7MXhAsGNptIQK0is2bWaXs2J6CHZB+eKv1/MxzKcuJSEXq2SYdm1RvurriRrVCyttXC9DlHxbqJf8h4RBA4Od8L5A2zGM3bpuYGqMq+sbfE5sLwLCzutWhvf1f6YB7hL8S5yiXzc1330x+455cbzEaLMtboM5TPFmGNm598ETu2ugSWcc44cqGAZ+VZJdSDYmGcXkYnNw2wh6thPbmQbE2wKxk7tpuHpydYqW5ZBWkOvDLYJ0L8ny5HQ9vltjqYIA23RBwC5oECB4FARGnEso6TzZG6g9CRPl+rIRMSH0lSN//MN+mWf7szjZj5Irtb0lry05og74upD3ZSJuBs5ckI3G+vF2rKupBjxlIkxqH3y1X6T7vB2on/q34WZt7yF58FqR/JaTQumPz0ZfDJegwMYFGJfNxyDAZlkQ9KsINiIbOMGIDat91cuzeQ/xcu6tdvOKL8ibrkGuLyZr6zaXehS8dqvLO+JouLniF6rrMNsMmc1NlBaHdzfhzbBIh3/96RTq8SrKwGXNq1vkWpaOu1Uat6Zou3pgfNQAx3ez/IGrUfQPhltdlDLA6SJSauu1aS5erobwoboq5Zy8KJzp+iidfpR9xrEJ9oC8oFIQeHm0Xg4QZ6ltUwH4qrzA1QRLkkXBqrTssVriV7BIrxPElKpGgOSbx1V2Ephj3GzFPsf5nqaJaEzMN30bcAusftn0nnRUNWX0yxE0mvATnhufu8FVJ14NMe98Ty8EnsG9fr/u3nhSCnxKZJouiKRmzvY4IUIX6yIY9PWCahMb6HF4/ejpQB0fPKd/Ryejp09PWj5TcwPzfO/XpLqcQ0c3eA+YMG6QXJKPHxy1BLxxZF3vOkvDigyEuiwLWmxbQr3RjV7cC16XvHnK6RiGoXTGtrATqrhj/KP6n9dRcjR8adJT+Qye3nKZIeQAbTdDEIGhv1A4Nhyuhz//oq4z7MFluQuNfIyNHJpbLJQZ2vmer94LkibSQz7JZUIkUBLXHuj9/Lxc0i8+nmV5JyRzAqEqGhhQoOHiwKq086ou+rUmVdTQQ6NBBWbgoEV+tH1eO1YhBEWP2Hn1kM7h5H848J1cDzvjfUkjr3OvhUkjR2knn/01uhUuW57vaQHhetJyfYueUPUZA/tDdvdCwbwFte81oQnk546jDG569p3/CCAvRXk7hwMBh1Q5A3eocDY+md7V5kWyPyo+N4ILjwIsjaoq727b540lU3VyZsiKuO4yI4LgmoLJ+d62ON8z/j79nG9LwgTBt2WD8JqYE7XETqjO+/AM2OaKri+1M/wLY9I8dllIH+4AOJH73+u7vdLgIqNB9GVLq9oh8hxRn+Eal4lRUVxS7SRy4T5NlbO75GPA/a7Ij3DWK5LcL7nQiLHXNV2swjX8nVslEF7wsco8ImsBOVYaD44g0pRzCzuzRvNpzYGflfQXNTgxGlDX4af1SjphxRmU7mWq61u5p9OWbBIKKR0+fIAJJ20ElnS+DpdLoHpfTgpEugijet9KujXpSYhHLMkiRcze7j3wcTCD4JhbcNdGJ45VU0lycQULVc5MbNQMjTRTNSoH9TwtM94MqlsPDn/gY48aIMDCLcypHXLPURZItpEvSqW2RJE4hIRabI1YOP3BPXOTXy0C3AZU1/g6CD2pkNFHfjAqYqNNoweSFsl0avI4XI3yazLHLj7C3kIRzsz/8LefXiD5kCBhMxobJyUZpzjE7qSdCRawIHXYeZhvQ6QQpLIV46iZYID/v7O5PBjWOr1GDrVnwHya1NkXejlekE2OWYW/m9k9R3zA+dLUHc9PNaT54jG2ZTmTxl6rts3SRTmP5ZIXfVXhLMxjVzOiLTTcu7N3fjijVe+H2QXSlg9wiMo9YPXOZqn1pz6auSR+p55MBJYThM7S38TEMfRgF7LFFg5gR8x3plK5zQ4Pjp81LS3bJdKISDGjzrTGQM5zwyjqe/WuTnBbvXqsT8TYDvyP1Mz+qrEAk9iOeQ03/MLHKuEMW2eMP1AlrozA5bDFFpJqBeZBBAnoxlvPHtHB59TFiPD5oSrSGX58K5EsUR9XnQTeab9ZSfoBnHrNH1xxACFHvY3UUCeWn9ZP6GBfoSJOvciB43o+lQ6aZfHiIvJF35IkfAeB9u6uZk1BVvpy0YGnO4L7cL5X7l7bO1fzC1Y4as03VaBTI4Amyc4tLfKUCG0ttgLefcPhExnqImAKTfjfQUWJINvkOr26gzqSxk+/M5ydc4bryHZEpc60HrLCdqAMezjTVjPqwcangy4ONfphMqmrt8ZS4gU4kLwmrMOIHCa+UdMTrpGEqa6YEbtT4P+aS8z65Szmc5HtNjCCYD+d70lt/zP/vfM3663fZU/i3NtAlVuyz6B0iUZ1LcUAv0ZCRUF/zvfqhNLHTFEo2h/ghKzlVv4CLFWctrv+o/pPtqSJX/bn6brcJCbg8P6XD7hpEJdS3Q7E/OS7X1Yhb7iuwiSBOMdNsS2ovIuBQ3XIpzOpiHVOGRlWUU9wh8fDlTCGHdUOTLUOTExqtU1JzS6IpIUcdoVBnNOKcxKg5SYdCbudjVtyjLgrMCqug5zMSD4PuyZ+8OTM4FIT1h04CxAiSg1Euf1CAEywGoh3LOjxHEK6Ba0DkhZVNMs6hwgxCZn6QB+yZzF31z9DkI35vA8lZYD7RcpuUAfsplyg9SLr0P8c3/mfwdB3lsn/FR6H+KgvsZ17heRlOli9evVGfYaKHd3tjw7RTWdwWge2rU2rfSZxiYhPKquz7i1pz8a2dm1gNHIf1KvyBbdS6UxMpISstSotHOAXAXyVDYwrsoR429R4CA7gaQNy1UYjjnReH+Jy7Tnn2aUbt4N+czT1GKIMqC1EG62qDGqDo7Q830OIU/MgOWP2cEN8E+EY3yq8iuX+x3UUZolc9qRzZm0eqxG9bgC/7ToalciMLxGn7g2Aht5v+p7WDVobP2jNKbBRwKeY+jbsolPdKtfIiJsOx8j1uR7aH3Q/7fkYd83GF8IUUyeQBHDG32SAd1lD/W5vzkka3hkm1+zmRMdf/Xw6xPY9zRZcAsuJUNjk2fC5EcnYw/2yK30dZXWPdk6ujI8gxHtbz+TnxTCq+FZDNwGLOlQ6zE6TVWbMJmM4Un/+2ynRzaQy4r7bEHd5z3D9Jq75TZfFlsQG9T8Pbwc6XIjbp+QS9SvSuryn76sPhVzCeZ68end0yJtEZ5yLMcBdVwt9RwdvPU75Rnc58IWQI2IU1VW74XoNv6ixD7Dh3RK5QPV269xe/jO13n35uqmVJI+9kr0K6pMR8Qb3ECSJ/0uKW/v6uwP7rFYHKtrmk1E0rAWe83KLMwq+bdEMQ2mbZUJI+u8ML+jbC3DVnsfA+81muJpvgh7d8DAJueswW2isblw8bqn2jVW+TzQlMjY+bnYYJ8PRgC8BxJ0ErDhIn5jO2zfXeLcE1hlSdRlFkVIXfH6mX+V/B/k8TAJwrPfJZJhLDIQ6bT64aj5gjmw8dCQAOLnZBAdb6s/6avht7YG5XUUfRpQiXEyQe94nH5sN7I/rr4fV1yP6emW+9m2w8pNvspTc00P6oFXrnJSU/hYXPl/nCBsf7H3mYoiD2cY/56vZKt9cozKqbewzndiGlEE9kScwooqpXyY04TiWUfdnjZd5JdXIVfAMnCfqsqutvh7i0AEiPiJ6Nm2/xsGJIuxqX8HttwoYG/ep2wtJFL6bCJOrAcPACLcRc6/+P7Ogcw2Gs5ZqU32NS/c8hjkwA9BnU502Kac8eoagn+4lCKmSDR8hwUKTFWF5pl8Hymcn9FEo9PHkvlE1bx6SQFdBikDv7Es+3cTsof9RvbwRWaivZZeNcva+0y3uJNxSF7hTE9eY35IZhogWX6A+50v8LmgQ6m1G/gb574AX3URzFv+IL+fVvX1WCUngNQovomxNa/UjUPzAvfZy9eOvpJFwPjLlPRgA5FtZr2O+rvcqvSTN+rf3H3+Us0G4D5Nq/piuwxm11jmN+/SX72DHCyLKeF3gclt91SQpkcCMkWVNbYVJiVcJob57eQ3Jvk8khWhND4zuE92ICTHLyybvp0lyg+T9OAnwNYiY0F7HLTN1hO7roza5rqi8r4mjvIxpQHr/8RpscvTsWPR1PnkqXwLw0NNj3ni94u3WyfEXqTeGNqi+AyDNhH5snjQa2c6wk2C+14OSHFgQh+qTeFGX8ueqvkMgMIkXC3fbtIpFoBER6Oc4Ievb04pc/5nFYS5niZsO5i6IQHa00DB/5B8NmHY4A3WRpSVZDPd2xuRYMqZiInyQdxqQue610DbUrAFzh1dVeLjTmdNpu5u81rbR9DRYpxfdFo2EnhkCJD3HBQMPAu1p36dWNVd+UV1VRmuqBkGMogoCy1i0P/APDkb9gXIejPqtwS2KAOg3cVwgSKXjxdR0aPvejLzhomjD4ZH6QZIGelOfl6UEEWwvXeUwpxqwLv4vzXIL6ZSvSaGOjIlrGrQr7mKBNvvcY4Yi0cY9XTyBxzRQT55cXiOk5a5WFOMUYK0+pzg2nshBBQASJ6yylwPcg+zdnPm+P1AH7l7Gp8sr50Klf0aQ/I7A4L3CpO/d9LXi3mVdPQTIyJC+98rfhgvvd59dZ/LSIiwUTv1tSMkhTIKDflNK0YCNPfGKbI512YbVt4bFjd7+OSAT88ZuBQ12S3G3N62WaFTUp7tDBt9zfSgMx2aCeBwwjm6e3PT1jUt9SQxzLkUjHU+abLN1IrSfAH2HB/Op7cHIfWVtB4b7q6CC3S53QL38p6Fe6IQVR3wTY9huvq7Lh9ZOqZES3tCRMgYXiMK+jwunPbEF2SQk3DsiIDTt2ofnOG7+r8y27tkxWTnpv7JjnEvsxDhx9IgzemxabtJ44fVrto4hm6HahW2ncQ7C3Es7oNQR8hwRR9VpRgG05oMSCBYndU9C21qO4JPm7aUsKUHyr3ue+TIKOKgnh1UkcaayFSVAZk5LIjTIoSRjqNU2fjQuNStO2ld71PKbp6rxmw2q6lFlQNldfY6FEBV8vctzubCvlftrvIl/itZFeGokbs0UbRd7TorepIkkZ/exPdlE1xqXbbwfiuQRG5JPEFxNRq5Z2h6sLq2U1ED2xIJ4cTPh6XmwqxCXxAdJdBGQ+ia/0MDhbCYwlnmAMB9yk1YltolrtvgP2Bl4lBHe5ht87jOxrwdqabNtmrzhbvg0NDtWx2hQfT90rAFcHFl9bzSUqPVkB7c3LRpZzo71QOjeZziW2y+qrrccqwZwZB5vlbjOHD5I77wxslnGVhHLQypgw45hACRC6yk3FVG9m7Z8sR0AeaOWLbW8pN83UbRyi1+VgOGwBILwcmeLF0NGNxxSeuYn8LD+MFGHyLXHA5aeeMKiTH65RcgfJ2F0cH9e1UuTKJuQ2IMNbgO8EhyYyl0y6vVPckYAndl0JPf+Vs8TK2+Cq0v+S9lfCJp03OPqCdZy+6j6lhp1XOLaRvcXQDWH9WcctHiB5TzUl0aZC6WQFptFenPeQbfqGwGY0Rh3Qplf4+Fo2nGpaQcWfN9VxgFe22+u7K1JVlJgfrFtEPAkew3mrG4sF2Ks0+SiupEV1p+2B2xwyBcZaR4YDT+0uT2GG8F1K2L0kb0XnY0dVnUjBzmdonsjR9TGNSDuXbba7ubjfx2X2lZS0SQRPlIuflHMQe5i4L2qejYdl8r7Zlhs2ExprV1rSeVVopyT/+Om1yU++teDF1xrmT6IHC4m5JAeNlGockZq4qujv0FLtu8Wh/ruHIYoNEX42DsTBSRiut/M6BLwdquzP21C1Vf0NH3GOlo7GnWMrqmkHhgYX/K0ezOXO+vYy9WLWFp/q54dBAcH/N9DRwKW53uaMzlD+zNDGA/uqm06ErrhDK98enbwi16/3RrGyueBHL3Ntd+rb6UTs0kPuBaxR10EPsrtOvIMQxOppASmI58e42o6A12cYHnWPkiu6yIdkWTWwuyld4K2t7RpYI8gmHsNnLQa63OPeT1nXKNTRWHjHJvlcyzAvrwByhY/IqiKz26lWNfrjWXnLFY06QjE67C/HMGonwzfnZLlLENI4QauwIMLrXyvl0N/V5S/59a0eGpGpNeZ11qpXoc26cqS4dMMDj/uONwqtJCObzqwnsC4uPG6eQYffvEbAgKn86LkV9Hx60iV5M/yTgGtq3WemlsS+dVemqkusnC7crNlzY5jY676reHIutDV3KXKm/M7FupkdGhvk4zYR+G4uFVy4vQ33hHiVO9aX/EGq+tLF5M5fEnrye2gvqp+18zgOePQ58n7fX3DXot0Wp//Xh+o5NqYcQrxftca3a033ZXtkMkNZTrfQbuU8gdXIOJ4Md5h3n2+z7z+uX4ssv2y9YePA776+TR4//Jvrz+8fvtmYE482JeoR/bFmLq6MBrukSyRGi5XorFkgqsnb7oMkEYozx44VXjv6UHWxYOaZz8wttDA8arcd4TM00znteqDyNV1uJqhzIZt95HmWqISjg7p1ncCVNKU5CvXtAll5+eJqUveCBtdd5Ioo18WN0YNbnnHKWb0bIJDrGT30NNGzpn1Y+SoZ3Wyxd4i8/l8T07bIiHHcIMvjwLdTLKPLKvUKuovUgWHpTlXiNWJOV+Kpy2v/3zvAmeK7D13OGaNA0Syec8Zb95B/xG3dbGCb0OfLUdPg4TzlBu9EBAutGvek5t5cMFitCnXznW27QwnDbijxYO9MIWW6zAwxx8DHPaLCzR0F89dlWngxMll5e1pGPlquGYLb2jdY31mJ4sL8x06vPqmxYSphsQwmje2QLtCMud7yVW8iMMhRjNMrrJiLk1bj+mrZDvuKDuqx/TbN9LgTQ3EiW2R5aMki2e8481pzQ1tKlxsj5GhuuG6Zhf69TlcBUo74DthoSVQEpTZ2pwXbalaqbLTNjirthEkb1MWtz20mnsCwE32kDsZOkb8jpZ+eBG9SYtXSAhjFTW+d9C4dQOqj9Txeg33tuYnSuVKEHASXBbxETq8Tjz3bDSUbAG2E7oTZzQSkiP2C7+Dvcqi4/xU2U8QgU7dX2DuJB2BXx+EHDt+0Sa/w1Hh5q8khRqz5881uS1CPh+w5HPGgpl+0HL/T3Mc3aeeTAjgz3p0TRT1Pdo2863K+oMhTIvQTr9+OWL9XSByEtaeXo4dAB13+qQzTu1ApoygL1NWM2EMrCpeZFrJk07jpTHcpTNeAWNvCzenXHELSr2rO3VFnla9s7qJE5LGg44yzWxgwTZqbaHoV/VMOl8GFOoYEpO6UTYz8SXXJVvzncJeqIYK5bPc7cqcQVHmfTpcX95ihPwf89XnHS289cLjLEHWKOaVQkZ9OI70PKQ1hLdZNADz2ywYrpwU499fCNwmzoCvoO4FNvnKnn9wQhLEH6knaiY99NsHiP6ofng1eiovyJY3vSHDBu+Av8BLP8OM3+DO7/TR74XVb9kUBeWrH1DBhTdL+WrvAneKsDnDc4NXx+K6wbz+ehOuYOlepPZwbgWPyWcXOgDhhaWRpCZxllG17Flk4fVCKSQstcwkVOceEIEGhdCGPIK/U78JvKrHL/zkdysJ8TEMz6KKd74cHMnrA6r51U8Rdly6sDo/PJvu/E3UAQOsdTI6caMoLEHsO3zMLRBjLUXO9/SlB2N5QV4oa1CMBB4FiuTb7i0TbGbpw1jrQ2Naji1W9zUkGgTEaVWjFrt3svN9ICvqVlCrZ/e13KbrGHdZjY2Bpc9EcOvzPTNLggYsIFZKgcNZ3VZMF5X4JlF05Uy/AKYJfOTwdsN4iEZV9pzBnrPrK94CA7gloYnA3Us/XgBsmfO3u3rIWXjRDQW/TSIJUZrs7IcjwhxG+aKYsERrdADmi7aF3I3hZvjGOmli0Mr1QBJq/VTG88t6eG/nTQDW49I3fGtnbYxXbvJphJ7pwDnc0bvrPNZz/xH+uRMYr53X0H7epLYlXXMdxUEz3hvmt/LkBu5WdlBxgN4dFV+j4zGH9prP234OLvrPC33A/nzv9P0vYuP//fSD+RbN/vzuV/m+DfnF5+auWHMchSDkckK/owd9vAXCXVa+XtYGohxD5hLeznnhXlEves6eaFCLdF7KqR00ZkZAU/7S0XdFOGRTSCecNave3XLS7As3d1bydXHVrty/AsPq7Zs3/0tHMc73tM/WvK1hM2+eWPpXD0zxOzqZgfinWWGyljdzZCTVXmgyswbH2WigjjmMzTSpbqEmYhjXcMrZEtzC7l/LTqOB8M1APT15BBTdykKR1Xa+xxfoQ89nzYM0vu93rq26vwgKMYagkka14/SudI86GpEO76167wjOFiVebWeVpM28e6/FfHTanHNopAHA9aKwe9gdCZbBfFu7Cr3ZzRm2FKGSRwa62t9X5JOBgw7a7Zw3ttjbUnaAF8/kgSC5+STabtU4TVsWag0y331tAVN97OIm04EZkvzc0dq8LBfvcLzwQ/Z4gvkaOXTamargcU98s9h6ckh2+EBl5uuXQ6/TwfjEO8+6nlkhN3VMPZtxNK6nG0E08YSjRLOLWStsCWrWcRU6lsJdzQt4eRNlcxjT0U2RkaGgD2RwAti+nKCwO+3VrdnACUrXMa9N+0k966nG+XohVElE9qdOHXIzYx7mO92lTcOTk8b2ugi+XsQbHh7gjovmGux3g5rV7m/YlOvAGx0cNPai3IUuYqwxSv3KipzRbh7ZlW1V3aF3U88/ocLWtQaG2K5JJGzWdH2p9T2O8Y0PdHQF1zzSvNfW/9NaovM9l392xri7LJGW6MYxkn0kmW75hauPE9y1fciOYG5npKP28hXCAOdXysS2eqEqLFRlr7hmqLnwx1KssqWm/OqeXeFUG8tzXmGnx6FPZ0vrqzn2S5eprqifSmQ3iIENojpufPcxI8W4NO4CymRfO0dMzVWB8Ld1+E9W4IzsV8TetsEcR3d8nP2uMY/RvP/auudZ9OrmBj7LkMO5TrYaG78wTiqDl4hOSLG5sgNMFVrT7ye0Z6HRQeNsNMLE86K5bHXcoBE2aO8pNAP1bnq7sUnbaw/mtexuaxOW7bkCJxrk1KUlyH+PzeJe38b7Mv8OI+YxQP8fWjX2nibz5jiDlsGc/x2oKIG8WMhPLMjO4Hbzo0OglhO99rEKk444YLa/p0K7B7OBqN8HHtpM9aGaVed4nXBj16UY95hJVcR2Vn1lI6nDXgczo0x0ISr5B0f/itXDC/zx9k4rGiU7nPdo8cMuLd5gzgfUeo379eU5Z2ej6RMCjD/HX9PEPTm6dyXZjZSOYy6SuIv9O2eTBXzSZDiu6LPzTJMNz1b4O8eNisirT6Kue8udFfHvXwBi84gA/xK2XuPl6xO2ijpPcjTGUPm4JkoixGjfSdJoCAJVJ7gxfjxhe8kDDjSFhPaTCoK9hL3pc9gpNLzMaPwebz2TGQhyOpCmA+m9lt1jd4zaW2xP7NfarHQZdKZmoEN2rFRM+G53bRvflBu/9V0v3O6sM0YsSWeSVxCbAKHLvmRilZkEDgwYfY1xV0tj3pmq1hqtb3cZqI/Z1qrv4vF7Xzn/gbvnq2dFvvi/pbRqDOS++vqBUD4+53t8C5pqUJt39NpBv4b15Kj/f5fRLdbQDnPbCTSxi/AlVrd+GY69X4UESSDXPUlygOsLOvk4ng0PS9B3oLwHHCiridmIkAqtAEjfvE7GXjItUyhp6zY91QTVoJfSLcnAZrJrrt/Q204TkhcO/LebXv/fhovs3vfDASMbWdKaJks3QVJutreenjaflpon6bmfK34YqxtfqvUr1hjLMQr9/A6K5wuNIZ3ZYPDfbQhxdLxuCY2i4fG/YgpVeTiPsIUeIb30Z+cWnh4t2ZFmuHUzsn255JlBciobDouU31fA2Iex7yQSyXIc37NSeVTpdZTZ4P3PaWouXeuM9T954mb/GfZw3wpSw/LfJnVlWJoYnbd8dSqDztd3cLbPgvgz9z5fjg3Cl6IsL3UOlxv87969aW7btPZr8ILXmDdAJofG+tC6996gEGHR2H7qOBjJasjmhHdrugfekOvsp/FJNNHZvnqd5GDFBzfX+vU3irRDGR0vFZmF+eo8Ie5Tw6hM1TbeRugW15t0v2CEXx4n95qoP7/71eh8X70z17GYO0844sQbQf558sOvb376+WXw/u3bj+QJ/A9vvlDDocI3gsxJYPLz8w+nH/4SfHj76/sfX5Kkwqj2oa3Vf/yH2l4v+lC7tq0DFM9/fPvz6Q/Bu398/MvbN+jks/tgPJS8zDvUpEK3jMAPNxi7SQUjYqief8Z0m/YeUb/XHQhT36uLuPh6VRTbfLy/T99X5cwnI3ef6g/T7GK/u51PNb9/fnJ8fPRsMVrOZyfRaDY7OJkfHi6fPz94toiez+nHYvnNcRid3IefvMSadN5+sdma3Och5z4PeXRDqjWUWn5xU3SCUn/6U+/dP3rNRGb9skKe4XHP2DCBSXwNgoHqgQeqMjeptW8BEBdVVf75XNbem7dvXvYqsMj3acDtCFf1+bUyj4mutt720/vAF2/SanDWAZaHLA55qTae8wtAX/18um+uUvwtneU+UH33D1qyd/8HyIx31A==')))
_patch_stamp = str(time.time_ns())
for _relative, _source in _patch_files.items():
    _target = BUNDLE / _relative
    _target.parent.mkdir(parents=True, exist_ok=True)
    if _target.exists() and _target.read_text() == _source:
        print(f'Already current: {_relative}')
        continue
    _backup = _target.with_name(_target.name + '.before-' + _patch_stamp)
    if _target.exists():
        _backup.write_bytes(_target.read_bytes())
    _temporary = _target.with_name(_target.name + '.update-' + _patch_stamp)
    _temporary.write_text(_source)
    _temporary.replace(_target)
    print(f'Updated {_relative}' + (f'; backup: {_backup.name}' if _backup.exists() else ' (new file)'))

def run(command, *args):
    import sys, time
    log_dir = CONFIG.parent / 'logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f'{command}-{time.time_ns()}.log'
    argv = [sys.executable, '-u', '-m', 'elastic_embed.cli', command, '--config', str(CONFIG), *args]
    print(f'Python: {sys.executable}\nLog: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f'{command} failed (exit {returncode}). See output above or {log_path}')

print('Patch applied. Rerun the failed command; detailed failures will be printed and logged.')


The default `table-ffn` mode retains projected Qwen token embeddings and selected FFNs. MOHAWK then aligns KDA mixers and complete blocks before whole-student recovery. The student is a causal KDA encoder. This is a width/depth-compressed adaptation of MOHAWK; it does not guarantee Qwen generalization. For a random-init control, explicitly set `mohawk.enabled: false` and `initialization.mode: random` before starting a separate run.

Use the new `smoke-mohawk` run directory. Old initialized checkpoints lack the bridge/layer provenance required by these stages; do not overwrite or silently migrate an existing optimizer checkpoint.


In [5]:
cfg = yaml.safe_load(CONFIG.read_text())
root = Path(cfg['run_dir'])
if not (root / 'initialized.pt').exists():
    run('initialize')


Python: /usr/bin/python3
Log: /content/drive/MyDrive/elastic_embed_colab/runs/logs/initialize-1790186747326834277.log
2026-09-23 18:05:55.258562: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-23 18:05:55.334314: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Transferred FFN 1/12
Transferred FFN 2/12
Transferred FFN 3/12
Transferred FFN 4/12
Transferred FFN 5/12
Transferred FFN 6/12
Transferred FFN 7/12
Transferred FFN 8/12
Transferred FFN 9/12
Transferred FFN 10/12
Transferred FFN 11/12
Transferred FFN 12/12


The default calibration cache uses 256 train + 32 held-out validation documents, contiguous prefixes up to 128 tokens, and 64-token head-mean attention maps. Five projected state streams plus maps take about 1.7 GB. This teacher pass uses eager attention to expose maps. The teacher exits before optimization starts.


In [6]:
if cfg.get('mohawk', {}).get('enabled', False):
    run('cache-mohawk', '--resume')


Python: /usr/bin/python3
Log: /content/drive/MyDrive/elastic_embed_colab/runs/logs/cache-mohawk-1790186792816875294.log
MOHAWK cache upper-bound arrays: 1.73 GB, 256 train / 32 validation
`torch_dtype` is deprecated! Use `dtype` instead!
2026-09-23 18:06:40.580842: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-23 18:06:40.651943: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.18s/it]
MOHAWK cache 8/288; 1.6s
MOHAWK cache 16/288; 2.9s
MOHAWK cach

Matrix orientation updates Q/K projections, their short convolutions, and decay/update gates. It compares normalized head-mean token-mixing matrices, because Qwen and KDA have different head counts. FP32 reference materialization is used only for this short warmup.


In [7]:
if cfg.get('mohawk', {}).get('enabled', False):
    run('train', '--stage', 'mohawk-matrix', '--resume')


Python: /usr/bin/python3
Log: /content/drive/MyDrive/elastic_embed_colab/runs/logs/train-1790186859627500439.log
2026-09-23 18:07:58.404119: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-23 18:07:58.478782: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
mohawk-matrix 1/50 loss=1.95526
mohawk-matrix 5/50 loss=0.91531
mohawk-matrix 10/50 loss=0.43995
mohawk-matrix 15/50 loss=0.26725
mohawk-matrix 20/50 loss=0.19576
mohawk-matrix 25/50 loss=0.16098
mohawk-matrix 30/50 loss=0.13321
mohawk-matrix 35/50 loss=0.11821

Independent block alignment gives each student block its mapped teacher input. It fits attention outputs, full block outputs, and normalization behavior. FFNs are frozen first, then can adjust at the transferred-parameter learning rate. Each checkpoint records held-out per-layer and per-language alignment.


In [8]:
if cfg.get('mohawk', {}).get('enabled', False):
    run('train', '--stage', 'mohawk-block', '--resume')


Python: /usr/bin/python3
Log: /content/drive/MyDrive/elastic_embed_colab/runs/logs/train-1790187574678377740.log
2026-09-23 18:19:55.005695: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-23 18:19:55.081898: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
mohawk-block 1/150 loss=8.43658
mohawk-block 5/150 loss=7.46247
mohawk-block 10/150 loss=6.48428
mohawk-block 15/150 loss=5.81950
mohawk-block 20/150 loss=5.51230
mohawk-block 25/150 loss=5.47990
mohawk-block 30/150 loss=5.41696
mohawk-block 35/150 loss=5.52103

In [9]:
if cfg['initialization']['mode'] != 'random':
    run('cache-recovery', '--resume')
    run('recover', '--resume')


Python: /usr/bin/python3
Log: /content/drive/MyDrive/elastic_embed_colab/runs/logs/cache-recovery-1790188155655935719.log
`torch_dtype` is deprecated! Use `dtype` instead!
2026-09-23 18:29:24.131910: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-23 18:29:24.206771: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.07s/it]
Recovery cache 32/2048
Recovery cache 64/2048
Recovery cache 96/2048
Recovery cache 128/2048
Recovery cache 160/2048
Recovery ca

In [10]:
for stage in ('mohawk-matrix', 'mohawk-block'):
    report = root / f'{stage}_report.json'
    if report.exists():
        print(stage, report.read_text())


mohawk-matrix {
  "stage": "mohawk-matrix",
  "cache_id": "48d335b08d95bbb2bf330960a5be22c9069f9e4374a967ad915b00cc49b0564f",
  "validation": [
    {
      "step": 0,
      "examples": 32,
      "coverage": {
        "en": 2,
        "es": 2,
        "pt": 2,
        "fr": 2,
        "de": 2,
        "ru": 2,
        "ar": 2,
        "hi": 2,
        "bn": 2,
        "zh": 1,
        "ja": 2,
        "ko": 2,
        "id": 2,
        "tr": 1,
        "vi": 1,
        "th": 1,
        "sw": 2,
        "te": 2
      },
      "per_language_loss": {
        "en": 1.9963013529777527,
        "es": 2.0017233788967133,
        "pt": 1.9616732994715371,
        "fr": 1.95668297012647,
        "de": 1.9866901288429897,
        "ru": 1.9805571188529334,
        "ar": 1.922021011511485,
        "hi": 1.9568682511647542,
        "bn": 1.987660159667333,
        "zh": 1.9650421837965648,
        "ja": 1.952926223476728,
        "ko": 1.9594623098770778,
        "id": 1.9427170604467392,
        "tr